# Verification of MELG generators from Harase & Kimoto (2017)

This notebook verifies the generators from Table I of:

> Shin Harase and Takamitsu Kimoto. *Implementing 64-bit Maximally
> Equidistributed $\mathbb{F}_2$-Linear Generators with Mersenne Prime
> Period.* ACM Trans. Math. Softw., 2017.

For each generator we check:
1. The characteristic polynomial has the correct Hamming weight $N_1$
2. The generator has full period $2^p - 1$ (irreducibility test)
3. With Lagged Tempering, the total dimension defect is $\Delta = 0$ (ME)

In [1]:
import time
from regpoly.generateur import Generateur
from regpoly.transformation import Transformation
from regpoly.combinaison import Combinaison
from regpoly.analyses.equidistribution_test import (
    EquidistributionTest, METHOD_MATRICIAL, METHOD_DUALLATTICE
)

INT_MAX = 2**31 - 1

## Table I parameters

All seven generators from the paper, with both the recurrence
parameters ($M$, $\sigma_1$, $\sigma_2$, $a$) and the tempering
parameters ($L$, $\sigma_3$, $b$).

In [2]:
TABLE_I = [
    {
        "name": "MELG607-64", "p": 607, "w": 64, "r": 33, "N": 10,
        "M": 5, "sigma1": 13, "sigma2": 35,
        "a": 0x81f1fd68012348bc, "N1": 313,
        "L": 3, "sigma3": 30, "b": 0x66edc62a6bf8c826,
    },
    {
        "name": "MELG1279-64", "p": 1279, "w": 64, "r": 1, "N": 20,
        "M": 7, "sigma1": 22, "sigma2": 37,
        "a": 0x1afefd1526d3952b, "N1": 641,
        "L": 5, "sigma3": 6, "b": 0x3a23d78e8fb5e349,
    },
    {
        "name": "MELG2281-64", "p": 2281, "w": 64, "r": 23, "N": 36,
        "M": 17, "sigma1": 36, "sigma2": 21,
        "a": 0x7cbe23ebca8a6d36, "N1": 1145,
        "L": 6, "sigma3": 6, "b": 0xe4e2242b6e15aebe,
    },
    {
        "name": "MELG4253-64", "p": 4253, "w": 64, "r": 35, "N": 67,
        "M": 29, "sigma1": 30, "sigma2": 20,
        "a": 0xfac1e8c56471d722, "N1": 2129,
        "L": 9, "sigma3": 5, "b": 0xcb67b0c18fe14f4d,
    },
    {
        "name": "MELG11213-64", "p": 11213, "w": 64, "r": 51, "N": 176,
        "M": 45, "sigma1": 33, "sigma2": 13,
        "a": 0xddbcd6e525e1c757, "N1": 5455,
        "L": 4, "sigma3": 5, "b": 0xbd2d1251e589593f,
    },
    {
        "name": "MELG19937-64", "p": 19937, "w": 64, "r": 31, "N": 312,
        "M": 81, "sigma1": 23, "sigma2": 33,
        "a": 0x5c32e06df730fc42, "N1": 9603,
        "L": 19, "sigma3": 16, "b": 0x6aede6fd97b338ec,
    },
    {
        "name": "MELG44497-64", "p": 44497, "w": 64, "r": 47, "N": 696,
        "M": 373, "sigma1": 37, "sigma2": 14,
        "a": 0x4fa9ca36f293c9a9, "N1": 19475,
        "L": 95, "sigma3": 6, "b": 0x06fbbee29aaefd91,
    },
]

print(f"{'Name':<16} {'p':>6} {'N':>4} {'r':>3} {'M':>4} "
      f"{'s1':>3} {'s2':>3} {'N1':>6} {'L':>4} {'s3':>3}")
print("-" * 65)
for g in TABLE_I:
    print(f"{g['name']:<16} {g['p']:>6} {g['N']:>4} {g['r']:>3} {g['M']:>4} "
          f"{g['sigma1']:>3} {g['sigma2']:>3} {g['N1']:>6} "
          f"{g['L']:>4} {g['sigma3']:>3}")

Name                  p    N   r    M  s1  s2     N1    L  s3
-----------------------------------------------------------------
MELG607-64          607   10  33    5  13  35    313    3  30
MELG1279-64        1279   20   1    7  22  37    641    5   6
MELG2281-64        2281   36  23   17  36  21   1145    6   6
MELG4253-64        4253   67  35   29  30  20   2129    9   5
MELG11213-64      11213  176  51   45  33  13   5455    4   5
MELG19937-64      19937  312  31   81  23  33   9603   19  16
MELG44497-64      44497  696  47  373  37  14  19475   95   6


## Check 1: Full period and Hamming weight $N_1$

For each generator, compute the characteristic polynomial via
Berlekamp-Massey and verify:
- Hamming weight matches $N_1$ from the paper
- The polynomial is irreducible (= primitive for Mersenne prime exponents)

**Note:** Larger generators take longer (BM is $O(k^2)$).

In [3]:
generators = {}

print(f"{'Name':<16} {'k':>6} {'N1 (got)':>10} {'N1 (paper)':>12} "
      f"{'Full period':>12} {'Time':>8}")
print("-" * 70)

for g in TABLE_I:
    t0 = time.time()

    gen = Generateur.create("MELG", g["w"],
        w=g["w"], N=g["N"], M=g["M"], r=g["r"],
        sigma1=g["sigma1"], sigma2=g["sigma2"], a=g["a"])

    cp = gen.char_poly()
    hw = bin(cp._val).count('1') + 1  # +1 for the leading x^k term

    fp = gen.is_full_period()

    elapsed = time.time() - t0

    hw_ok = "OK" if hw == g["N1"] else f"FAIL (got {hw})"
    fp_ok = "OK" if fp else "FAIL"

    print(f"{g['name']:<16} {gen.k:>6} {hw:>10} {g['N1']:>12} "
          f"{fp_ok:>12} {elapsed:>7.1f}s")

    generators[g["name"]] = gen

Name                  k   N1 (got)   N1 (paper)  Full period     Time
----------------------------------------------------------------------
MELG607-64          607        313          313           OK     0.0s
MELG1279-64        1279        641          641           OK     0.0s
MELG2281-64        2281       1145         1145           OK     0.1s
MELG4253-64        4253       2129         2129           OK     0.4s
MELG11213-64      11213       5455         5455           OK     3.5s
MELG19937-64      19937       9603         9603           OK    14.5s
MELG44497-64      44497      19475        19475           OK   104.6s


## Check 2: Equidistribution with Lagged Tempering ($\Delta = 0$)

Apply the paper's Lagged Tempering parameters and verify $\Delta = 0$
(maximally equidistributed).

- Small generators ($p \leq 4253$): use the matricial method
- Large generators ($p \geq 11213$): use the dual lattice method

In [5]:
print(f"{'Name':<16} {'k':>6} {'se':>6} {'ME':>4} {'Method':<10} {'Time':>8}")
print("-" * 58)

for g in TABLE_I:
    gen = generators[g["name"]]
    w = g["w"]

    trans = Transformation.create("laggedTempering",
        w=w, sigma=g["sigma3"], L=g["L"], b=g["b"])

    comb = Combinaison(J=1, Lmax=w)
    comb.components[0].add_gen(gen)
    comb.components[0].add_trans(trans)
    comb.reset()

    #method = METHOD_MATRICIAL if g["p"] <= 4253 else METHOD_DUALLATTICE
    method = METHOD_DUALLATTICE
    method_name = "matricial" if method == METHOD_MATRICIAL else "lattice"

    t0 = time.time()
    test = EquidistributionTest(L=w, delta=[INT_MAX] * (w + 1),
                                mse=INT_MAX, method=method)
    result = test.run(comb)
    elapsed = time.time() - t0

    me = "yes" if result.is_me() else "no"
    se_ok = "OK" if result.se == 0 else f"FAIL (se={result.se})"

    print(f"{g['name']:<16} {gen.k:>6} {result.se:>6} {me:>4} "
          f"{method_name:<10} {elapsed:>7.1f}s")

Name                  k     se   ME Method         Time
----------------------------------------------------------
MELG607-64          607      0  yes lattice        0.2s
MELG1279-64        1279      0  yes lattice        0.4s
MELG2281-64        2281      0  yes lattice        0.8s
MELG4253-64        4253      0  yes lattice        1.8s
MELG11213-64      11213      0  yes lattice        7.2s
MELG19937-64      19937      0  yes lattice       19.9s
MELG44497-64      44497      0  yes lattice      107.6s


## Check 3: Equidistribution without tempering

For comparison, show the dimension defect without tempering.
Only computed for the smaller generators (matricial method).

In [6]:
print(f"{'Name':<16} {'k':>6} {'se (raw)':>10} {'se (tempered)':>14}")
print("-" * 50)

for g in TABLE_I:
    if g["p"] > 4253:
        continue
    gen = generators[g["name"]]
    w = g["w"]

    # Without tempering
    comb_raw = Combinaison(J=1, Lmax=w)
    comb_raw.components[0].add_gen(gen)
    comb_raw.reset()

    test = EquidistributionTest(L=w, delta=[INT_MAX] * (w + 1),
                                mse=INT_MAX, method=METHOD_MATRICIAL)
    result_raw = test.run(comb_raw)

    print(f"{g['name']:<16} {gen.k:>6} {result_raw.se:>10} {'0 (ME)':>14}")

Name                  k   se (raw)  se (tempered)
--------------------------------------------------
MELG607-64          607        431         0 (ME)
MELG1279-64        1279        887         0 (ME)
MELG2281-64        2281        941         0 (ME)
MELG4253-64        4253        481         0 (ME)


## Detailed gaps table for MELG19937-64

Full dimension gaps table for the flagship generator, with and
without tempering.

In [7]:
g = TABLE_I[5]  # MELG19937-64
gen = generators[g["name"]]
w = g["w"]

# With tempering
trans = Transformation.create("laggedTempering",
    w=w, sigma=g["sigma3"], L=g["L"], b=g["b"])

comb = Combinaison(J=1, Lmax=w)
comb.components[0].add_gen(gen)
comb.components[0].add_trans(trans)
comb.reset()

print(f"MELG19937-64 with Lagged Tempering")
print(f"  sigma={g['sigma3']}, L={g['L']}, b=0x{g['b']:016x}")
print()

test = EquidistributionTest(L=w, delta=[INT_MAX] * (w + 1),
                            mse=INT_MAX, method=METHOD_DUALLATTICE)
result = test.run(comb)

print(f"Dimension gaps sum = {result.se}")
if result.is_me():
    print("==> MAXIMALLY EQUIDISTRIBUTED")
print()

table_str, _ = result.display_table(comb, 'l')
print(table_str)

MELG19937-64 with Lagged Tempering
  sigma=16, L=19, b=0x6aede6fd97b338ec

Dimension gaps sum = 0
==> MAXIMALLY EQUIDISTRIBUTED


=======+=====+=====+=====+=====+=====+=====+=====+=====+=====+=====+=====+=====+=====+=====+=====+=====+
RESOL  |    1|    2|    3|    4|    5|    6|    7|    8|    9|   10|   11|   12|   13|   14|   15|   16|
-------+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----|
ECART  |     |     |     |     |     |     |     |     |     |     |     |     |     |     |     |     |
-------+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----|
DIM    |19937| 9968| 6645| 4984| 3987| 3322| 2848| 2492| 2215| 1993| 1812| 1661| 1533| 1424| 1329| 1246|
=======+=====+=====+=====+=====+=====+=====+=====+=====+=====+=====+=====+=====+=====+=====+=====+=====
=======+=====+=====+=====+=====+=====+=====+=====+=====+=====+=====+=====+=====+=====+=====+=====+=====+
RESOL  |   17|   18|   19|   20